# Fine-tune V-JEPA2 (SSv2) on SAILS rmm_type

This notebook mirrors the structure of `scripts/notebook_finetuning.ipynb` but targets the SAILS clips generated from the CSV folds. It uses the `rmm_type` column as labels and assumes the clips live under `/orcd/scratch/bcs/001/sensein/sails/rmm/vjepa2_finetune_clips/<csv_stem>/<segment_id>.mp4` as produced by `dataprep/create_clip_segments.py` with deduplication.

## Setup
The environment should already have `torch`, `numpy`, `decord`, and `transformers` (>=4.44) installed. Adjust the paths below if your clip or CSV locations differ.


In [ ]:
from pathlib import Path
import csv
from functools import partial
from typing import List, Dict

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from decord import VideoReader, cpu

from transformers import VJEPA2ForVideoClassification, VJEPA2VideoProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
csv_dir = Path("/orcd/data/satra/001/users/brukew/actreg/dataprep/cv_folds")
clips_root = Path("/orcd/scratch/bcs/001/sensein/sails/rmm/vjepa2_finetune_clips")

# Pick which folds to use for train/val
train_csvs = ["fold_0_train.csv", "fold_1_train.csv", "fold_2_train.csv"]
val_csvs = ["fold_0_val.csv", "fold_1_val.csv", "fold_2_val.csv"]
cur_train, cur_val = [train_csvs[0]], [val_csvs[0]]

model_id = "facebook/vjepa2-vitl-fpc16-256-ssv2"

proj_name = "vjepa-rmm"

csv_dir, clips_root, model_id


## Build split manifests from CSV + clip folders
Each CSV row should have `segment_id` (or `segment_global_id`), `rmm_type`, and the clip should exist in `<clips_root>/<csv_stem>/<segment_id>.mp4`. This cell collects train/val records and checks for missing clips.

In [ ]:
def load_split(csv_names: List[str]) -> List[Dict]:
    records = []
    missing = []
    for name in csv_names:
        csv_path = csv_dir / name
        stem = csv_path.stem
        with csv_path.open("r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fieldnames = [fn.strip().replace("\ufeff", "") for fn in reader.fieldnames]
            reader.fieldnames = fieldnames
            for row in reader:
                seg = row.get("segment_id") or row.get("segment_global_id")
                label = row.get("rmm_type")
                if not seg or label is None:
                    continue
                clip_path = clips_root / stem / f"{seg}.mp4"
                if not clip_path.exists():
                    missing.append((seg, clip_path))
                    continue
                records.append({"segment_id": seg, "label": label, "clip": clip_path})
    print(f"Loaded {len(records)} records from {len(csv_names)} CSVs; missing clips: {len(missing)}")
    if missing:
        print("Example missing:", missing[:3])
    return records

train_records = load_split(cur_train)
val_records = load_split(cur_val)
all_labels = sorted({r["label"] for r in train_records})
label2id = {lbl: i for i, lbl in enumerate(all_labels)}
id2label = {i: lbl for lbl, i in label2id.items()}
len(train_records), len(val_records), label2id

In [ ]:
processor = VJEPA2VideoProcessor.from_pretrained(model_id)

frames_per_clip = (
    getattr(processor, "num_frames", None)
    or getattr(getattr(processor, "image_processor", processor), "num_frames", None)
    or getattr(getattr(processor, "feature_extractor", processor), "num_frames", None)
    or getattr(getattr(processor, "config", {}), "num_frames", None)
    or getattr(getattr(processor, "config", {}), "frames_per_clip", None)
    or 16
)
frames_per_clip = 32
frames_per_clip


## Dataset + DataLoaders
We sample frames with Decord using simple even spacing (one clip per video) and let the processor handle normalization/resizing.


In [ ]:
class RMMDataset(Dataset):
    def __init__(self, records, label2id, frames_per_clip):
        self.records = records
        self.label2id = label2id
        self.frames_per_clip = frames_per_clip

    def __len__(self):
        return len(self.records)

    def _sample_indices(self, vr):
        total = len(vr)
        if total <= 0:
            return np.zeros(self.frames_per_clip, dtype=np.int64)
        return np.round(np.linspace(0, total - 1, self.frames_per_clip)).astype("int64")

    def __getitem__(self, idx):
        rec = self.records[idx]
        try:
            vr = VideoReader(str(rec["clip"]), ctx=cpu(0))
            indices = self._sample_indices(vr)
            frames = vr.get_batch(indices).asnumpy()  # (T, H, W, C) uint8
        except Exception as e:
            print(f"[bad clip] {rec['clip']}: {e}")
            return None
        label_id = self.label2id[rec["label"]]
        return frames, label_id


def collate_fn(samples, processor):
    samples = [s for s in samples if s is not None]
    if not samples:
        return None, None
    frame_batches, labels = zip(*samples)
    inputs = processor(list(frame_batches), return_tensors="pt")
    labels = torch.tensor(labels)
    return inputs, labels

train_ds = RMMDataset(train_records, label2id, frames_per_clip)
val_ds = RMMDataset(val_records, label2id, frames_per_clip)

batch_size = 1
num_workers = 8

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=partial(collate_fn, processor=processor),
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=num_workers > 0,
    prefetch_factor=2 if num_workers > 0 else None,
)
val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=partial(collate_fn, processor=processor),
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=num_workers > 0,
    prefetch_factor=2 if num_workers > 0 else None,
)

batch0 = next(iter(train_loader))
batch0[0] if batch0[0] is None else batch0[0]['pixel_values_videos'].shape, len(train_ds), len(val_ds)


## Initialize model for classification (rmm_type head)
We freeze the V-JEPA backbone and train only the classification head.

In [ ]:
model = VJEPA2ForVideoClassification.from_pretrained(
    model_id,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
).to(device)

for param in model.vjepa2.parameters():
    param.requires_grad = False

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable, lr=1e-5)
num_epochs = 10
accumulation_steps = 4

model.config

## Training + simple eval
Gradient accumulation simulates a larger batch size. Evaluation reports accuracy on the validation set.

In [ ]:
import wandb
run_name = f"base-whole-video-{frames_per_clip}fr"
wandb.init(project=proj_name, name=run_name, config={"lr": 1e-5, "batch_size": 1, "frames": frames_per_clip})

In [ ]:
def evaluate(loader, model, device):
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            if inputs is None or labels is None:
                continue
            labels = labels.to(device)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = logits.argmax(-1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = correct / max(total, 1)
    return acc, all_preds, all_labels


def class_names_from_id2label(id2label):
    return [id2label[i] for i in range(len(id2label))]


for epoch in range(1, num_epochs + 1):
    model.train()
    optimizer.zero_grad()
    running_loss = 0.0
    num_batches = 0

    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{num_epochs}")
    print("=" * 60)

    for step, (inputs, labels) in enumerate(train_loader, start=1):
        if inputs is None or labels is None:
            continue

        labels = labels.to(device)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss / accumulation_steps
        loss.backward()
        running_loss += loss.item()
        num_batches += 1

        if step % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        if step % 50 == 0:
            avg_loss = running_loss / num_batches * accumulation_steps
            print(f"  Step {step}/{len(train_loader)} | Loss: {avg_loss:.4f}")

        wandb.log({"train/loss": loss, "epoch": epoch, "step": step})

    if num_batches % accumulation_steps != 0:
        optimizer.step()
        optimizer.zero_grad()

    avg_loss = running_loss / max(num_batches, 1) * accumulation_steps
    val_acc, val_preds, val_labels = evaluate(val_loader, model, device)
    wandb.log({"val/acc": val_acc, "epoch": epoch})

    if val_labels:
        cm = wandb.plot.confusion_matrix(
            preds=val_preds,
            y_true=val_labels,
            class_names=class_names_from_id2label(id2label),
        )
        wandb.log({"val/conf_mat": cm, "epoch": epoch})

    print(f"\n>>> Epoch {epoch} complete: avg_loss={avg_loss:.4f}, val_acc={val_acc:.3f}")

output_dir = Path("runs/vjepa2_rmm_type")
output_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)
output_dir
wandb.finish()

## Top-K Evaluation
Analyze top-2 predictions to understand model performance on semantically similar classes (e.g., "hands flapping" vs "one hand flap").

In [ ]:
from tqdm import tqdm

def evaluate_topk(loader, model, device, k=2):
    """Evaluate with top-K accuracy and collect predictions for analysis."""
    model.eval()
    correct_top1 = 0
    correct_topk = 0
    total = 0

    all_preds_top1 = []
    all_preds_topk = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc="Evaluating", total=len(loader)):
            if inputs is None or labels is None:
                continue

            labels = labels.to(device)
            inputs = {k_input: v.to(device) for k_input, v in inputs.items()}
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1)

            # Top-1
            preds_top1 = logits.argmax(-1)
            correct_top1 += (preds_top1 == labels).sum().item()

            # Top-K
            topk_values, topk_indices = torch.topk(probs, k, dim=-1)
            correct_topk += torch.any(topk_indices == labels.unsqueeze(-1), dim=-1).sum().item()

            total += labels.size(0)

            # Collect for analysis
            all_preds_top1.extend(preds_top1.cpu().tolist())
            all_preds_topk.append(topk_indices.cpu())
            all_labels.extend(labels.cpu().tolist())
            all_probs.append(probs.cpu())

    top1_acc = correct_top1 / max(total, 1)
    topk_acc = correct_topk / max(total, 1)

    # Concatenate tensors
    all_preds_topk = torch.cat(all_preds_topk, dim=0) if all_preds_topk else torch.tensor([])
    all_probs = torch.cat(all_probs, dim=0) if all_probs else torch.tensor([])

    return {
        'top1_acc': top1_acc,
        f'top{k}_acc': topk_acc,
        'improvement': topk_acc - top1_acc,
        'preds_top1': all_preds_top1,
        'preds_topk': all_preds_topk,  # Shape: (N, K)
        'labels': all_labels,
        'probs': all_probs,  # Shape: (N, num_classes)
    }


def analyze_top2_confusion(preds_top1, preds_topk, labels, id2label):
    """Analyze cases where top-1 is wrong but top-2 saves it."""
    import numpy as np
    from collections import Counter

    preds_top1 = np.array(preds_top1)
    preds_top2 = preds_topk[:, 1].numpy() if isinstance(preds_topk, torch.Tensor) else np.array(preds_topk)[:, 1]
    labels = np.array(labels)

    # Find "saves" - wrong top-1, correct top-2
    wrong_top1 = preds_top1 != labels
    if wrong_top1.sum() == 0:
        print("\n✓ Top-2 'Saves' Analysis: All top-1 predictions correct!")
        return

    correct_top2 = preds_top2[wrong_top1] == labels[wrong_top1]

    print(f"\nTop-2 'Saves' Analysis:")
    print(f"  Total wrong top-1: {wrong_top1.sum()} ({100*wrong_top1.sum()/len(labels):.1f}% of samples)")
    print(f"  Saved by top-2: {correct_top2.sum()} ({100*correct_top2.sum()/max(wrong_top1.sum(),1):.1f}% of errors)")

    # Most common saves
    save_pairs = []
    for i in np.where(wrong_top1 & (preds_top2 == labels))[0]:
        true_label = id2label[labels[i]]
        wrong_pred = id2label[preds_top1[i]]
        save_pairs.append((true_label, wrong_pred))

    if save_pairs:
        print("\n  Most common 'saves' (true label ← mistaken as):")
        for (true, wrong), count in Counter(save_pairs).most_common(5):
            print(f"    • {true} ← {wrong}: {count} times")

# Run evaluation
print("Running top-K evaluation on validation set...")
val_topk_metrics = evaluate_topk(val_loader, model, device, k=2)

print(f"\n{'='*60}")
print("TOP-K EVALUATION RESULTS")
print(f"{'='*60}")
print(f"Top-1 Accuracy: {val_topk_metrics['top1_acc']:.3f}")
print(f"Top-2 Accuracy: {val_topk_metrics['top2_acc']:.3f}")
print(f"Improvement:    +{val_topk_metrics['improvement']:.3f} ({100*val_topk_metrics['improvement']:.1f}% gain)")

analyze_top2_confusion(
    val_topk_metrics['preds_top1'],
    val_topk_metrics['preds_topk'],
    val_topk_metrics['labels'],
    id2label
)

### Confusion Analysis: Where does Top-2 help?
Visualize confusion patterns and see which class pairs benefit most from top-2 predictions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Prepare data
preds_top1 = np.array(val_topk_metrics['preds_top1'])
preds_topk = val_topk_metrics['preds_topk'].numpy()
labels = np.array(val_topk_metrics['labels'])
probs = val_topk_metrics['probs'].numpy()

# Create confusion matrix for top-1
cm = confusion_matrix(labels, preds_top1)
class_names = [id2label[i] for i in range(len(id2label))]

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Standard confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, 
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_xlabel('Predicted (Top-1)')
axes[0].set_ylabel('True Label')
axes[0].set_title('Confusion Matrix: Top-1 Predictions')
axes[0].tick_params(axis='x', rotation=45)

# Right: Top-2 "save" rate per class
wrong_mask = preds_top1 != labels
save_rate_per_class = []

for class_id in range(len(id2label)):
    class_mask = labels == class_id
    wrong_in_class = wrong_mask & class_mask
    
    if wrong_in_class.sum() > 0:
        # Of the wrong predictions for this class, how many are saved by top-2?
        saved = (preds_topk[wrong_in_class, 1] == labels[wrong_in_class]).sum()
        save_rate = saved / wrong_in_class.sum()
    else:
        save_rate = 0.0
    
    save_rate_per_class.append(save_rate)

axes[1].barh(class_names, save_rate_per_class, color='steelblue')
axes[1].set_xlabel('Top-2 Save Rate (% of errors recovered)')
axes[1].set_title('Top-2 Recovery Rate by Class')
axes[1].set_xlim([0, 1])
for i, v in enumerate(save_rate_per_class):
    axes[1].text(v + 0.02, i, f'{v:.1%}', va='center')

plt.tight_layout()
plt.savefig('topk_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Print per-class top-1 accuracy
print("\nPer-Class Top-1 Accuracy:")
print(f"{'Class':<20} {'Accuracy':>10} {'Count':>8}")
print("-" * 40)
for class_id in range(len(id2label)):
    class_mask = labels == class_id
    if class_mask.sum() > 0:
        acc = (preds_top1[class_mask] == labels[class_mask]).mean()
        print(f"{id2label[class_id]:<20} {acc:>9.1%} {class_mask.sum():>8}")


### Log Top-K Analysis to W&B
Save all metrics, confusion matrices, and visualizations to Weights & Biases for tracking.

In [ ]:
# Initialize W&B run for top-K analysis
topk_run = wandb.init(
    project=proj_name,
    name=f"{run_name}-topk-analysis",
    job_type="evaluation",
    config={
        "model_checkpoint": str(output_dir),
        "k": 2,
        "frames_per_clip": frames_per_clip,
    }
)

# Log top-K metrics
wandb.log({
    "eval/top1_accuracy": val_topk_metrics['top1_acc'],
    "eval/top2_accuracy": val_topk_metrics['top2_acc'],
    "eval/top2_improvement": val_topk_metrics['improvement'],
    "eval/top2_improvement_pct": 100 * val_topk_metrics['improvement'],
})

# Log confusion matrix
cm_wandb = wandb.plot.confusion_matrix(
    probs=None,
    preds=val_topk_metrics['preds_top1'],
    y_true=val_topk_metrics['labels'],
    class_names=class_names,
)
wandb.log({"eval/confusion_matrix": cm_wandb})

# Log the visualization
wandb.log({"eval/topk_visualization": wandb.Image("topk_analysis.png")})

# Create and log per-class metrics table
class_metrics_data = []
for class_id in range(len(id2label)):
    class_mask = labels == class_id
    if class_mask.sum() > 0:
        top1_acc = (preds_top1[class_mask] == labels[class_mask]).mean()
        
        # Top-2 accuracy for this class
        class_preds_topk = preds_topk[class_mask]
        class_labels = labels[class_mask]
        top2_correct = np.any(class_preds_topk == class_labels[:, None], axis=1).sum()
        top2_acc = top2_correct / class_mask.sum()
        
        # Save rate (of errors, how many saved by top-2?)
        wrong_in_class = (preds_top1[class_mask] != labels[class_mask])
        if wrong_in_class.sum() > 0:
            saved = (preds_topk[class_mask][wrong_in_class, 1] == labels[class_mask][wrong_in_class]).sum()
            save_rate = saved / wrong_in_class.sum()
        else:
            save_rate = 0.0
        
        class_metrics_data.append([
            id2label[class_id],
            class_mask.sum(),
            f"{top1_acc:.3f}",
            f"{top2_acc:.3f}",
            f"{save_rate:.3f}",
        ])

class_table = wandb.Table(
    columns=["Class", "Samples", "Top-1 Acc", "Top-2 Acc", "Top-2 Save Rate"],
    data=class_metrics_data
)
wandb.log({"eval/per_class_metrics": class_table})

# Log save pairs (confusion pairs that top-2 recovers)
preds_top2 = preds_topk[:, 1]
wrong_top1_mask = preds_top1 != labels
save_pairs_data = []

for i in np.where(wrong_top1_mask & (preds_top2 == labels))[0]:
    save_pairs_data.append([
        id2label[labels[i]],  # True label
        id2label[preds_top1[i]],  # Wrong top-1 prediction
        id2label[preds_top2[i]],  # Correct top-2 (same as true label)
        f"{probs[i, labels[i]]:.3f}",  # Probability of true label
    ])

if save_pairs_data:
    save_table = wandb.Table(
        columns=["True Label", "Wrong Top-1", "Correct Top-2", "True Prob"],
        data=save_pairs_data[:100]  # Limit to 100 for readability
    )
    wandb.log({"eval/top2_saves": save_table})

print(f"\n✓ Logged top-K analysis to W&B: {wandb.run.url}")

wandb.finish()

## Load Saved Model
Load the fine-tuned model from disk for inference or further training.

In [ ]:
model = loaded_model

In [ ]:
# Option 1: Load from local directory (default)
checkpoint_dir = Path("runs/vjepa2_rmm_type")

# Load the model
loaded_model = VJEPA2ForVideoClassification.from_pretrained(checkpoint_dir).to(device)
# loaded_processor = VJEPA2VideoProcessor.from_pretrained(checkpoint_dir)

print(f"✓ Loaded model from: {checkpoint_dir}")
print(f"  Classes: {loaded_model.config.id2label}")

# Test inference on a single sample
loaded_model.eval()
test_batch = next(iter(val_loader))
if test_batch[0] is not None:
    inputs, labels = test_batch
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        logits = loaded_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
        pred_class = logits.argmax(-1).item()
        confidence = probs[0, pred_class].item()
    
    print(f"\nTest inference:")
    print(f" Datapoint: {}")
    print(f"  Predicted: {loaded_model.config.id2label[pred_class]} ({confidence:.2%} confidence)")
    print(f"  True label: {loaded_model.config.id2label[labels[0].item()]}")
    print(f"\n  All class probabilities:")
    for class_id, prob in enumerate(probs[0].tolist()):
        print(f"    {loaded_model.config.id2label[class_id]:<20} {prob:.2%}")


### Inference on New Video Clips
Use the loaded model to classify new video clips.

In [ ]:
def predict_video(video_path, model, processor, device, top_k=2):
    """
    Run inference on a single video file.
    
    Args:
        video_path: Path to video file
        model: Fine-tuned VJEPA2 model
        processor: VJEPA2VideoProcessor
        device: torch device
        top_k: Number of top predictions to return
    
    Returns:
        dict with predictions and probabilities
    """
    from decord import VideoReader, cpu
    import numpy as np
    
    # Load video
    try:
        vr = VideoReader(str(video_path), ctx=cpu(0))
        total_frames = len(vr)
        
        # Sample frames (same as training)
        indices = np.round(np.linspace(0, total_frames - 1, frames_per_clip)).astype("int64")
        frames = vr.get_batch(indices).asnumpy()  # (T, H, W, C)
        
    except Exception as e:
        return {"error": str(e)}
    
    # Preprocess
    inputs = processor([frames], return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Inference
    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]  # (num_classes,)
        
        # Top-K predictions
        topk_probs, topk_indices = torch.topk(probs, top_k)
    
    results = {
        "video": str(video_path),
        "total_frames": total_frames,
        "predictions": []
    }
    
    for rank, (prob, idx) in enumerate(zip(topk_probs.tolist(), topk_indices.tolist()), 1):
        results["predictions"].append({
            "rank": rank,
            "class": model.config.id2label[idx],
            "class_id": idx,
            "probability": prob,
        })
    
    return results


# Example: Predict on a validation clip
example_clip = val_records[0]["clip"]
result = predict_video(example_clip, loaded_model, loaded_processor, device, top_k=3)

print(f"Video: {result['video'].split('/')[-1]}")
print(f"Total frames: {result['total_frames']}")
print(f"\nTop-{len(result['predictions'])} Predictions:")
for pred in result["predictions"]:
    print(f"  {pred['rank']}. {pred['class']:<20} {pred['probability']:.2%}")

# Batch prediction example
print("\n" + "="*60)
print("Batch prediction on first 5 validation clips:")
print("="*60)

for i, rec in enumerate(val_records[:5], 1):
    result = predict_video(rec["clip"], loaded_model, loaded_processor, device, top_k=2)
    if "error" in result:
        print(f"{i}. ERROR: {result['error']}")
        continue
    
    top1 = result["predictions"][0]
    top2 = result["predictions"][1]
    true_label = rec["label"]
    
    correct_marker = "✓" if top1["class"] == true_label else ("✓✓" if top2["class"] == true_label else "✗")
    
    print(f"{i}. {rec['segment_id']}")
    print(f"   True: {true_label:<20} | Pred: {top1['class']:<20} ({top1['probability']:.1%}) {correct_marker}")
    print(f"                              Top-2: {top2['class']:<20} ({top2['probability']:.1%})")
